# Customer Risk EDA and Cleaning Notebook

This notebook documents the cleaning work behind the dashboard: schema standardization, missing-value checks, duplicate checks, type conversion, numeric validation, feature engineering, and dashboard-ready risk segmentation.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

data_path = Path('../data/customer_risk.csv')
df = pd.read_csv(data_path)
df.head()


In [ ]:
# Initial quality checks
quality_summary = pd.DataFrame({
    'metric': ['rows', 'columns', 'duplicate_claim_ids', 'missing_cells'],
    'value': [len(df), df.shape[1], df.duplicated(subset=['claim_id']).sum(), df.isna().sum().sum()]
})
quality_summary


In [ ]:
# Cleaning workflow
clean = df.copy()
clean.columns = [c.strip().lower().replace(' ', '_') for c in clean.columns]
clean = clean.drop_duplicates(subset=['claim_id']).copy()

for col in ['claim_id', 'customer_id', 'region', 'policy_type', 'customer_age_band', 'claim_status']:
    clean[col] = clean[col].astype(str).str.strip().replace({'': np.nan, 'nan': np.nan})

clean['claim_date'] = pd.to_datetime(clean['claim_date'], errors='coerce')
for col in ['tenure_years', 'annual_premium', 'claim_amount']:
    clean[col] = pd.to_numeric(clean[col], errors='coerce')

clean = clean.dropna(subset=['claim_id', 'customer_id', 'claim_date', 'region', 'policy_type', 'customer_age_band', 'claim_amount'])
clean = clean[(clean['claim_amount'] >= 0) & (clean['annual_premium'] >= 0)].copy()
clean['region'] = clean['region'].str.title()
clean['policy_type'] = clean['policy_type'].str.title()
clean['claim_status'] = clean['claim_status'].str.title()
clean['loss_ratio'] = np.where(clean['annual_premium'] > 0, clean['claim_amount'] / clean['annual_premium'], np.nan)
clean['claim_month'] = clean['claim_date'].dt.to_period('M').astype(str)
clean.head()


In [ ]:
# Customer-level risk segmentation
claim_threshold = 4
loss_threshold = 15000
customer = clean.groupby('customer_id', as_index=False).agg(
    region=('region', lambda x: x.mode().iloc[0] if not x.mode().empty else x.iloc[0]),
    policy_type=('policy_type', lambda x: x.mode().iloc[0] if not x.mode().empty else x.iloc[0]),
    customer_age_band=('customer_age_band', lambda x: x.mode().iloc[0] if not x.mode().empty else x.iloc[0]),
    claims_count=('claim_id', 'nunique'),
    total_loss=('claim_amount', 'sum'),
    avg_loss=('claim_amount', 'mean'),
    annual_premium=('annual_premium', 'mean')
)
customer['risk_category'] = np.select(
    [(customer['claims_count'] >= claim_threshold) | (customer['total_loss'] >= loss_threshold),
     (customer['claims_count'] >= 2) | (customer['total_loss'] >= loss_threshold * 0.5)],
    ['High-risk', 'Medium-risk'],
    default='Low-risk'
)
customer['risk_category'].value_counts()


In [ ]:
# EDA summaries for dashboard validation
display(clean.groupby('customer_age_band')['claim_id'].nunique())
display(customer.groupby('risk_category').agg(customers=('customer_id', 'nunique'), total_loss=('total_loss', 'sum')))
display(clean.groupby('claim_month').agg(claims=('claim_id', 'nunique'), total_loss=('claim_amount', 'sum')).head())
